# 📥 Step 2: Enhanced Data Collection v2
## IntegriCheck — Academic Integrity Platform

### 🎯 Is Notebook Ka Kaam Kya Hai?
Yeh notebook **training data collect** karti hai — jitna zyada aur diverse data hoga, utna accurate hamara model banega.

### 📊 Dataset Targets (v1 → v2)
| Source | Pehle (v1) | Ab (v2) |
|--------|-----------|---------|
| Arxiv Papers | 500 | **2,400+** |
| Wikipedia Articles | 300 | **1,500+** |
| Student Essays | ❌ | **~25,000** |
| AI Detection Data | ~20k | **~50k+** |

### ⏱️ Expected Runtime
- Total: **30–60 minutes** (depends on internet speed)
- **Resume support** hai — agar beech mein band ho jaaye, dobara chalaoge toh wahi se shuru hoga

### ▶️ Run Order
Cells ko **upar se neeche order mein** run karo. Koi bhi step skip mat karo.

## Cell 1: Libraries Import + Path Setup
**Kya karta hai:** Saari zaroori Python libraries import karta hai aur folder structure banata hai.

- `os, sys, json` — file system operations
- `pandas, numpy` — data handling
- `tqdm` — progress bars (download track karne ke liye)
- `Path` — modern file paths

**Output:** Saare folders create ho jaayenge agar exist nahi karte.

In [1]:
# CELL 1 — Imports aur paths setup
# Kyu: Saare folders aur project paths initialize karne ke liye
# Expected output: Saare paths print honge + folders create ho jayenge

import os, sys, json, time, csv, random
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# ── Project root ka path set karo ─────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

RAW_DIR       = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

ARXIV_DIR     = os.path.join(RAW_DIR, 'arxiv')
WIKI_DIR      = os.path.join(RAW_DIR, 'wikipedia')
AI_DIR        = os.path.join(RAW_DIR, 'ai_datasets')
STUDENT_DIR   = os.path.join(RAW_DIR, 'student_essays')

# Agar folders exist nahi karte toh create karo
for d in [ARXIV_DIR, WIKI_DIR, AI_DIR, STUDENT_DIR, PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Paths print karo ──────────────────────────────────────────────────────────
print('Project paths:')
for name, path in [
    ('Root', PROJECT_ROOT),
    ('Raw', RAW_DIR),
    ('arXiv', ARXIV_DIR),
    ('Wiki', WIKI_DIR),
    ('AI data', AI_DIR),
    ('Student Essays', STUDENT_DIR),
    ('Processed', PROCESSED_DIR)
]:
    print(f'  {name:<16}: {path}')

print("\n" + "=" * 60)
print("  IntegriCheck — Enhanced Data Collection v2")
print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

print("\n✅ All folders ready!")

Project paths:
  Root            : e:\ADD\integricheck_project
  Raw             : e:\ADD\integricheck_project\data\raw
  arXiv           : e:\ADD\integricheck_project\data\raw\arxiv
  Wiki            : e:\ADD\integricheck_project\data\raw\wikipedia
  AI data         : e:\ADD\integricheck_project\data\raw\ai_datasets
  Student Essays  : e:\ADD\integricheck_project\data\raw\student_essays
  Processed       : e:\ADD\integricheck_project\data\processed

  IntegriCheck — Enhanced Data Collection v2
  Started: 2026-05-11 22:22:46

✅ All folders ready!


## Cell 2: Arxiv Paper Collector Function
**Kya karta hai:** Ek reusable function banata hai jo Arxiv se research papers download karta hai.

### Key Features:
- **Resume support** — pehle se downloaded files ko skip karta hai
- **Quality filter** — sirf woh papers jo 100+ words ka abstract ho
- **Rate limiting** — `time.sleep(0.12)` taaki Arxiv server block na kare
- Papers ko JSON format mein save karta hai

### Kyun Arxiv?
Arxiv mein actual academic research papers hain — exactly woh type ka content jo students plagiarize karte hain. Hamara corpus jitna research-paper-heavy hoga, plagiarism detection utni accurate hogi.

In [2]:
import arxiv

def collect_arxiv_papers(category, max_results=400, save_name=None):
    """
    Arxiv se ek specific category ke papers download karta hai.
    
    Parameters:
        category   : Arxiv category code (e.g., 'cs.AI', 'cs.CL')
        max_results: Kitne papers chahiye
        save_name  : Save file ka naam
    
    Returns: List of paper dictionaries
    """
    save_name = save_name or f"{category.replace('.','_')}_papers.json"
    save_path = os.path.join(ARXIV_DIR, save_name)

    # ── Resume support ─────────────────────────────────────────────────────────
    if os.path.exists(save_path):
        with open(save_path, 'r', encoding='utf-8') as f:
            existing = json.load(f)
        if len(existing) >= max_results * 0.8:  # 80% ho toh enough maano
            print(f"  ✓ {category}: Already have {len(existing)} papers — skipping")
            return existing

    papers = []
    print(f"\n  📥 Downloading {category} ({max_results} papers)...")

    search = arxiv.Search(
        query=f"cat:{category}",          # Category filter
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,  # Latest papers pehle
    )

    try:
        for result in tqdm(search.results(), total=max_results,
                           desc=f"  {category}", ncols=70):
            
            abstract = (result.summary or '').strip()
            title    = (result.title or '').strip()

            # Quality filter: abstract mein kam se kam 100 words chahiye
            if len(abstract.split()) < 100:
                continue

            papers.append({
                'doc_id':   result.entry_id.split('/')[-1],
                'title':    title,
                'text':     f"{title}. {abstract}",  # Title + abstract = corpus text
                'abstract': abstract,
                'category': category,
                'source':   'arxiv',
                'url':      result.entry_id,
                'date':     str(result.published)[:10],
                'authors':  ', '.join(str(a) for a in result.authors[:3]),
            })
            
            time.sleep(0.12)  # Polite rate limiting — server block se bachao

        # Save to JSON
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(papers, f, ensure_ascii=False, indent=2)
        print(f"  ✓ {category}: Saved {len(papers)} papers → {save_path}")

    except Exception as e:
        print(f"  ⚠ {category} failed: {e}")

    return papers

print("✅ Arxiv collector function ready!")


✅ Arxiv collector function ready!


## Cell 3: Arxiv Papers Download (6 Categories)
**Kya karta hai:** 6 different Arxiv categories se 2,400 papers download karta hai.

### Categories Kyun Choose Kiye?
| Category | Naam | Kyun Important? |
|----------|------|-----------------|
| `cs.AI` | Artificial Intelligence | Core ML/AI papers |
| `cs.CL` | Computation & Language | NLP papers — jo students copy karte hain |
| `cs.IR` | Information Retrieval | Search engines, IR papers |
| `stat.ML` | Machine Learning Stats | Statistical ML papers |
| `cs.CV` | Computer Vision | Image processing papers **NEW** |
| `cs.LG` | Learning/Deep Learning | Deep learning papers **NEW** |

**Estimate:** ~400 papers × 6 categories = **~2,400 total papers**

⚠️ **Agar internet slow hai** — cells mein max_results kam karo (e.g., 200)

In [3]:
# 6 categories define karo — naam aur kitne papers chahiye
ARXIV_CATEGORIES = {
    'cs.AI':   ('cs_AI_papers.json',   400),   # Artificial Intelligence
    'cs.CL':   ('cs_CL_papers.json',   400),   # Computation & Language (NLP)
    'cs.IR':   ('cs_IR_papers.json',   400),   # Information Retrieval
    'stat.ML': ('stat_ML_papers.json', 400),   # Machine Learning (Stats)
    'cs.CV':   ('cs_CV_papers.json',   400),   # Computer Vision (NEW)
    'cs.LG':   ('cs_LG_papers.json',   400),   # Learning / Deep Learning (NEW)
}

print("\n[1/4] 📚 ARXIV PAPERS COLLECTION")
print("-" * 45)

all_arxiv = []

# Har category ke liye papers download karo
for cat, (fname, n) in ARXIV_CATEGORIES.items():
    papers = collect_arxiv_papers(cat, max_results=n, save_name=fname)
    all_arxiv.extend(papers)

# Saare papers ek combined file mein save karo
all_arxiv_path = os.path.join(ARXIV_DIR, 'all_papers.json')
with open(all_arxiv_path, 'w', encoding='utf-8') as f:
    json.dump(all_arxiv, f, ensure_ascii=False, indent=2)

print(f"\n{'='*45}")
print(f"  ✅ Total Arxiv Papers Downloaded: {len(all_arxiv):,}")
print(f"  📁 Saved to: {all_arxiv_path}")
print(f"{'='*45}")



[1/4] 📚 ARXIV PAPERS COLLECTION
---------------------------------------------
  ✓ cs.AI: Already have 397 papers — skipping
  ✓ cs.CL: Already have 394 papers — skipping
  ✓ cs.IR: Already have 395 papers — skipping
  ✓ stat.ML: Already have 387 papers — skipping
  ✓ cs.CV: Already have 398 papers — skipping
  ✓ cs.LG: Already have 393 papers — skipping

  ✅ Total Arxiv Papers Downloaded: 2,364
  📁 Saved to: e:\ADD\integricheck_project\data\raw\arxiv\all_papers.json


## Cell 4: Wikipedia Article Collector Functions
**Kya karta hai:** Wikipedia se articles download karne ke liye do functions banata hai.

### Functions:
1. **`fetch_article(title)`** — Single article fetch karo
   - Stub articles skip karta hai (< 200 words)
   - First 5000 characters save karta hai (enough for corpus)

2. **`fetch_category_members(cat_title)`** — Wikipedia category ke saare articles fetch karo
   - Recursively subcategories bhi scan karta hai
   - Ek hi request mein 30-50 related articles milte hain

### Kyun Wikipedia?
- Students Wikipedia se often directly copy karte hain
- Broad academic knowledge base = better coverage
- Free aur open access

### Wikipedia API Setup:
`user_agent` mandatory hai — otherwise Wikipedia block kar deta hai

In [4]:
import wikipediaapi

def get_wiki_api():
    """Wikipedia API client create karo."""
    return wikipediaapi.Wikipedia(
        language='en',
        user_agent='IntegriCheck/2.0 (Academic Research; contact@integricheck.in)'
        # User agent zaroori hai — Wikipedia bina iske block karta hai
    )

def fetch_article(wiki, title, articles, seen):
    """
    Single Wikipedia article fetch karo.
    
    Returns: article dict ya None (agar exist nahi karta / stub hai)
    """
    if title in seen:
        return None
    seen.add(title)
    
    try:
        page = wiki.page(title)
        if not page.exists():
            return None
        
        text = page.text
        if len(text.split()) < 200:   # Stub articles skip karo
            return None
        
        return {
            'doc_id':   f"wiki_{len(articles):05d}",
            'title':    page.title,
            'text':     text[:5000],    # First 5000 chars = enough for corpus
            'source':   'wikipedia',
            'url':      page.fullurl,
            'category': 'general',
        }
    except Exception:
        return None  # Failed articles gracefully skip karo


def fetch_category_members(wiki, cat_title, depth=1, limit=50):
    """
    Wikipedia category ke saare articles fetch karo.
    depth=2 → subcategories bhi scan karta hai.
    """
    if depth == 0:
        return []
    try:
        cat = wiki.page(f"Category:{cat_title}")
        members = []
        for title, member in list(cat.categorymembers.items())[:limit]:
            if member.ns == 0:    # ns=0 = normal article
                members.append(title)
            elif member.ns == 14 and depth > 1:  # ns=14 = subcategory
                sub = title.replace('Category:', '')
                members.extend(fetch_category_members(wiki, sub, depth-1, limit//2))
        return members
    except Exception:
        return []

print("✅ Wikipedia helper functions ready!")


✅ Wikipedia helper functions ready!


## Cell 5: Wikipedia Download (1,500 Articles)
**Kya karta hai:** 1,500 Wikipedia articles download karta hai — academic topics pe focus.

### Topic Categories:
- **CS/AI/ML:** Machine learning, Deep learning, BERT, GPT, etc.
- **Data Science/Stats:** Regression, Bayesian inference, PCA, etc.
- **Academic/Research:** Plagiarism, Citation, Peer review, etc.
- **Medical/Bio:** Skin cancer, Melanoma, Clinical trials (skin disease project ke liye)
- **General Academic:** Climate change, Quantum computing, etc.

### Strategy:
Sirf direct article fetch nahi karta — category crawling bhi karta hai taaki related articles automatically mil jaayein. Ek topic se 10-30 related articles milte hain.

In [5]:
# Academic aur research-focused topics
WIKI_TOPICS = [
    # CS / AI / ML
    "Machine learning", "Deep learning", "Natural language processing",
    "Computer vision", "Neural network", "Artificial intelligence",
    "Reinforcement learning", "Transfer learning", "Transformer (machine learning)",
    "Support vector machine", "Random forest", "Gradient boosting",
    "Convolutional neural network", "Recurrent neural network", "BERT (language model)",
    "GPT (language model)", "Large language model", "Generative adversarial network",

    # Data Science / Statistics
    "Data science", "Statistics", "Regression analysis",
    "Hypothesis testing", "Bayesian inference", "Principal component analysis",
    "Cluster analysis", "Decision tree", "Overfitting", "Cross-validation",
    "Feature engineering", "Dimensionality reduction",

    # Academic / Research domains
    "Plagiarism", "Academic integrity", "Scientific method",
    "Research methodology", "Systematic review", "Meta-analysis",
    "Citation", "Peer review", "Open access", "Academic publishing",

    # Medical / Biology (skin disease project relevance)
    "Skin cancer", "Melanoma", "Medical imaging", "Dermatology",
    "Bioinformatics", "Genomics", "Clinical trial", "Epidemiology",

    # General academic subjects
    "Climate change", "Quantum computing", "Blockchain technology",
    "Cybersecurity", "Internet of things", "Autonomous vehicles",
    "Renewable energy", "Robotics",

    # Wikipedia categories (ek request mein 30+ related articles)
    "Machine learning algorithms", "Artificial intelligence applications",
    "Statistical methods", "Data mining", "Information retrieval",
    "Computational linguistics",
]

MAX_WIKI = 1500  # Target articles
save_path = os.path.join(WIKI_DIR, 'wikipedia_articles.json')

# Resume check
if os.path.exists(save_path):
    with open(save_path, 'r', encoding='utf-8') as f:
        existing_wiki = json.load(f)
    if len(existing_wiki) >= MAX_WIKI * 0.8:
        print(f"✓ Wikipedia: Already have {len(existing_wiki)} articles — skipping!")
        all_wiki = existing_wiki
    else:
        print(f"Resuming... already have {len(existing_wiki)}, need {MAX_WIKI}")
        all_wiki = existing_wiki
else:
    all_wiki = []

if len(all_wiki) < MAX_WIKI * 0.8:
    wiki    = get_wiki_api()
    seen    = set(a['title'] for a in all_wiki)  # Already downloaded track karo

    print(f"\n[2/4] 📖 WIKIPEDIA COLLECTION (target: {MAX_WIKI})")
    print("-" * 45)

    with tqdm(total=MAX_WIKI, initial=len(all_wiki),
              desc="  Wikipedia", ncols=70) as pbar:
        for topic in WIKI_TOPICS:
            if len(all_wiki) >= MAX_WIKI:
                break

            # Direct article fetch
            art = fetch_article(wiki, topic, all_wiki, seen)
            if art:
                all_wiki.append(art)
                pbar.update(1)

            # Category members fetch (related articles)
            try:
                members = fetch_category_members(wiki, topic, depth=2, limit=30)
                for title in members:
                    if len(all_wiki) >= MAX_WIKI:
                        break
                    art = fetch_article(wiki, title, all_wiki, seen)
                    if art:
                        all_wiki.append(art)
                        pbar.update(1)
                    time.sleep(0.05)
            except Exception:
                pass

            time.sleep(0.1)  # Rate limiting

    # Save
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(all_wiki, f, ensure_ascii=False, indent=2)

print(f"\n{'='*45}")
print(f"  ✅ Total Wikipedia Articles: {len(all_wiki):,}")
print(f"  📁 Saved to: {save_path}")
print(f"{'='*45}")


Resuming... already have 1182, need 1500

[2/4] 📖 WIKIPEDIA COLLECTION (target: 1500)
---------------------------------------------


  Wikipedia:  81%|█████████████   | 1220/1500 [04:26<32:42,  7.01s/it]


  ✅ Total Wikipedia Articles: 1,220
  📁 Saved to: e:\ADD\integricheck_project\data\raw\wikipedia\wikipedia_articles.json


## Cell 6: HC3 Dataset Load (Human vs ChatGPT)
**Kya karta hai:** HC3 (Human-ChatGPT Comparison) dataset load karta hai HuggingFace se.

### HC3 Dataset Kya Hai?
- **Full name:** Hello-SimpleAI/HC3
- **Content:** Ek hi question ke do answers — ek human ne likha, ek ChatGPT ne
- **Size:** ~60,000 Q&A pairs
- **Use:** AI detection model ko train karne ke liye — model seekhega ki human vs AI mein kya difference hai

### Labels:
- `label = 0` → Human written
- `label = 1` → AI (ChatGPT) written

### Domains covered:
Finance, Medicine, Law, Psychology, Open-domain QA

In [11]:
# CELL 3 — ADVANCED AI/HUMAN DATASET COLLECTION
# Kyu: Industry-level AI detection ke liye large balanced dataset banana
# Expected output:
#   - hc3_dataset.csv
#   - dolly_dataset.csv
#   - combined_ai_detection_dataset.csv

from datasets import load_dataset
import pandas as pd
import os
from tqdm import tqdm

# =========================================================
# Helper Function
# =========================================================

def clean_text(text):
    """Basic text cleaning"""
    
    if not isinstance(text, str):
        return ""
    
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = ' '.join(text.split())

    return text.strip()


# =========================================================
# HC3 DATASET
# =========================================================

def load_hc3_dataset(limit_per_class=5000):

    save_path = os.path.join(AI_DIR, 'hc3_dataset.csv')

    # Resume support
    if os.path.exists(save_path):

        df = pd.read_csv(save_path)

        if len(df) > 1000:
            print(f"  ✓ HC3 already exists ({len(df):,} rows)")
            return df

    print("\n  📥 Loading HC3 dataset...")

    rows = []

    try:

        # IMPORTANT:
        # datasets==2.14.6 use karo
        ds = load_dataset("Hello-SimpleAI/HC3", "all")

        human_count = 0
        ai_count = 0

        for split in ds.keys():

            for item in tqdm(ds[split],
                             desc=f"  HC3/{split}",
                             ncols=70):

                question = clean_text(item.get('question', ''))

                # HUMAN ANSWERS
                for ans in item.get('human_answers', []):

                    ans = clean_text(ans)

                    if len(ans.split()) < 40:
                        continue

                    rows.append({
                        'text': ans,
                        'label': 0,
                        'source': 'hc3_human',
                        'question': question[:200]
                    })

                    human_count += 1

                    if human_count >= limit_per_class:
                        break

                # AI ANSWERS
                for ans in item.get('chatgpt_answers', []):

                    ans = clean_text(ans)

                    if len(ans.split()) < 40:
                        continue

                    rows.append({
                        'text': ans,
                        'label': 1,
                        'source': 'hc3_chatgpt',
                        'question': question[:200]
                    })

                    ai_count += 1

                    if ai_count >= limit_per_class:
                        break

                if human_count >= limit_per_class and ai_count >= limit_per_class:
                    break

        df = pd.DataFrame(rows)

        df = df.drop_duplicates(subset=['text'])

        df.to_csv(save_path, index=False)

        print(f"  ✓ HC3 saved: {len(df):,} rows")

        return df

    except Exception as e:

        print(f"  ⚠ HC3 failed: {e}")

        return pd.DataFrame(columns=['text', 'label', 'source'])


# =========================================================
# DOLLY DATASET
# =========================================================

def load_dolly_dataset(limit=8000):

    save_path = os.path.join(AI_DIR, 'dolly_dataset.csv')

    # Resume support
    if os.path.exists(save_path):

        df = pd.read_csv(save_path)

        if len(df) > 1000:
            print(f"  ✓ Dolly already exists ({len(df):,} rows)")
            return df

    print("\n  📥 Loading Dolly dataset...")

    try:

        ds = load_dataset("databricks/databricks-dolly-15k")

        rows = []

        for item in tqdm(ds['train'],
                         desc="  Dolly",
                         ncols=70):

            response = clean_text(item.get('response', ''))

            if len(response.split()) < 40:
                continue

            rows.append({
                'text': response,
                'label': 1,
                'source': 'dolly_ai'
            })

            if len(rows) >= limit:
                break

        df = pd.DataFrame(rows)

        df = df.drop_duplicates(subset=['text'])

        df.to_csv(save_path, index=False)

        print(f"  ✓ Dolly saved: {len(df):,} rows")

        return df

    except Exception as e:

        print(f"  ⚠ Dolly failed: {e}")

        return pd.DataFrame(columns=['text', 'label', 'source'])


# =========================================================
# HUMAN DATA FROM YOUR EXISTING DATA
# =========================================================

def create_human_dataset():

    rows = []

    print("\n  📥 Creating human dataset from arXiv + Wikipedia...")

    # -----------------------------------------------------
    # arXiv
    # -----------------------------------------------------

    arxiv_path = os.path.join(ARXIV_DIR, 'all_papers.json')

    if os.path.exists(arxiv_path):

        with open(arxiv_path, 'r', encoding='utf-8') as f:
            papers = json.load(f)

        for p in papers:

            text = clean_text(p.get('text', ''))

            if len(text.split()) < 80:
                continue

            rows.append({
                'text': text,
                'label': 0,
                'source': 'arxiv'
            })

    # -----------------------------------------------------
    # Wikipedia
    # -----------------------------------------------------

    wiki_path = os.path.join(WIKI_DIR, 'wikipedia_articles.json')

    if os.path.exists(wiki_path):

        with open(wiki_path, 'r', encoding='utf-8') as f:
            articles = json.load(f)

        for a in articles:

            text = clean_text(a.get('text', ''))

            if len(text.split()) < 80:
                continue

            rows.append({
                'text': text,
                'label': 0,
                'source': 'wikipedia'
            })

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(subset=['text'])

    print(f"  ✓ Human dataset created: {len(df):,} rows")

    return df


# =========================================================
# MAIN EXECUTION
# =========================================================

print("\n[3/4] 🤖 ADVANCED AI DETECTION DATASETS")
print("-" * 50)

# HUMAN
df_human = create_human_dataset()

# AI
df_hc3   = load_hc3_dataset(limit_per_class=5000)
df_dolly = load_dolly_dataset(limit=8000)

# =========================================================
# COMBINE ALL
# =========================================================

all_dfs = [
    df_human,
    df_hc3,
    df_dolly
]

combined_df = pd.concat(all_dfs, ignore_index=True)

# Remove duplicates
combined_df = combined_df.drop_duplicates(subset=['text'])

# Remove very short texts
combined_df = combined_df[
    combined_df['text'].str.split().str.len() >= 40
]

# Shuffle dataset
combined_df = combined_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# =========================================================
# BALANCE DATASET
# =========================================================

min_class = combined_df['label'].value_counts().min()

df_human_balanced = combined_df[
    combined_df['label'] == 0
].sample(min_class, random_state=42)

df_ai_balanced = combined_df[
    combined_df['label'] == 1
].sample(min_class, random_state=42)

combined_df = pd.concat([
    df_human_balanced,
    df_ai_balanced
])

combined_df = combined_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\n✅ Dataset balanced!")
print(combined_df['label'].value_counts())



# Save final dataset
combined_path = os.path.join(
    PROCESSED_DIR,
    'combined_ai_detection_dataset.csv'
)

combined_df.to_csv(combined_path, index=False)

# =========================================================
# FINAL STATS
# =========================================================

print("\n" + "="*55)
print("  ✅ FINAL DATASET SUMMARY")
print("="*55)

print(f"\nTotal Samples: {len(combined_df):,}")

print("\nLabel Distribution:")
print(combined_df['label'].value_counts())

print("\nSource Distribution:")
print(combined_df['source'].value_counts())

print(f"\n📁 Saved to:")
print(combined_path)

print("\n✅ Advanced dataset ready for training!")


[3/4] 🤖 ADVANCED AI DETECTION DATASETS
--------------------------------------------------

  📥 Creating human dataset from arXiv + Wikipedia...
  ✓ Human dataset created: 3,069 rows

  📥 Loading HC3 dataset...
  ⚠ HC3 failed: Dataset scripts are no longer supported, but found HC3.py
  ✓ Dolly already exists (6,534 rows)

✅ Dataset balanced!
label
0    3069
1    3069
Name: count, dtype: int64

  ✅ FINAL DATASET SUMMARY

Total Samples: 6,138

Label Distribution:
label
0    3069
1    3069
Name: count, dtype: int64

Source Distribution:
source
dolly_ai     3069
arxiv        1944
wikipedia    1125
Name: count, dtype: int64

📁 Saved to:
e:\ADD\integricheck_project\data\processed\combined_ai_detection_dataset.csv

✅ Advanced dataset ready for training!


## Cell 7: GPT-Wiki + RAID Datasets
**Kya karta hai:** Do aur AI detection datasets load karta hai.

### GPT-wiki-intro:
- Wikipedia-style intros — human written vs GPT generated
- Same topic, different writing style
- Strong signal kyunki AI ki "encyclopedia style" pakdi jaati hai

### RAID Dataset:
- **Most comprehensive** AI detection dataset
- Multiple AI models: GPT-4, Claude, Llama, Mistral, etc.
- Multiple domains: news, reviews, abstracts, recipes
- Isse model seekhega ki sirf ChatGPT nahi, baaki AI models bhi detect karo

Agar RAID fail ho toh fallback: `ChatGPT-Research-Abstracts` dataset use hota hai.

In [13]:
# CELL — ADVANCED MULTI-SOURCE AI DETECTION DATASET LOADER
# Kyu:
#   - Stable dataset loading
#   - Better cleaning
#   - Duplicate removal
#   - Balanced AI/Human data
#   - Industry-level dataset pipeline
#
# Expected output:
#   - GPT Wiki dataset
#   - Research abstracts dataset
#   - Master balanced dataset

from datasets import load_dataset
import pandas as pd
import numpy as np
import os
import re
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

MAX_TEXT_LENGTH = 3000
MIN_WORDS = 40

TARGET_PER_CLASS = 50000

# =========================================================
# CLEANING FUNCTION
# =========================================================

def clean_text(text):

    if not isinstance(text, str):
        return ""

    text = text.replace("\n", " ")
    text = text.replace("\r", " ")

    # Remove URLs
    text = re.sub(r"http\\S+", " ", text)

    # Remove emails
    text = re.sub(r"\\S+@\\S+", " ", text)

    # Remove extra spaces
    text = re.sub(r"\\s+", " ", text)

    # Remove broken unicode
    text = text.encode("ascii", "ignore").decode()

    return text.strip()


# =========================================================
# GPT-WIKI DATASET
# =========================================================

def load_gpt_wiki_dataset(limit_per_class=50000):

    save_path = os.path.join(
        AI_DIR,
        'gpt_wiki_dataset.csv'
    )

    # -----------------------------------------------------
    # Resume Support
    # -----------------------------------------------------

    if os.path.exists(save_path):

        df = pd.read_csv(save_path)

        required_cols = ['text', 'label', 'source']

        if all(col in df.columns for col in required_cols):

            print(
                f"  ✓ GPT-Wiki already exists "
                f"({len(df):,} rows)"
            )

            return df

    print("\n  📥 Loading GPT-Wiki dataset...")

    try:

        ds = load_dataset(
            "aadityaubhat/GPT-wiki-intro"
        )

        rows = []

        for split in ds.keys():

            for item in tqdm(
                ds[split],
                desc=f"  GPT-Wiki/{split}",
                ncols=70
            ):

                # -----------------------------------------
                # HUMAN
                # -----------------------------------------

                wiki_text = clean_text(
                    item.get('wiki_intro', '')
                )

                if len(wiki_text.split()) >= MIN_WORDS:

                    rows.append({
                        'text': wiki_text[:MAX_TEXT_LENGTH],
                        'label': 0,
                        'source': 'gptwiki_human'
                    })

                # -----------------------------------------
                # AI
                # -----------------------------------------

                gpt_text = clean_text(
                    item.get('generated_intro', '')
                )

                if len(gpt_text.split()) >= MIN_WORDS:

                    rows.append({
                        'text': gpt_text[:MAX_TEXT_LENGTH],
                        'label': 1,
                        'source': 'gptwiki_ai'
                    })

        # -------------------------------------------------
        # DataFrame
        # -------------------------------------------------

        df = pd.DataFrame(rows)

        # Remove duplicates
        df = df.drop_duplicates(
            subset=['text']
        )

        # Balance
        human_df = df[df['label'] == 0]
        ai_df    = df[df['label'] == 1]

        human_df = human_df.sample(
            min(limit_per_class, len(human_df)),
            random_state=42
        )

        ai_df = ai_df.sample(
            min(limit_per_class, len(ai_df)),
            random_state=42
        )

        df = pd.concat([
            human_df,
            ai_df
        ])

        # Shuffle
        df = df.sample(
            frac=1,
            random_state=42
        ).reset_index(drop=True)

        # Save
        df.to_csv(save_path, index=False)

        print(
            f"  ✓ GPT-Wiki saved: "
            f"{len(df):,} rows"
        )

        return df

    except Exception as e:

        print(f"  ⚠ GPT-Wiki failed: {e}")

        return pd.DataFrame(
            columns=['text', 'label', 'source']
        )


# =========================================================
# RESEARCH ABSTRACTS DATASET
# =========================================================

def load_research_dataset(limit_per_class=25000):

    save_path = os.path.join(
        AI_DIR,
        'research_dataset.csv'
    )

    # -----------------------------------------------------
    # Resume Support
    # -----------------------------------------------------

    if os.path.exists(save_path):

        df = pd.read_csv(save_path)

        required_cols = ['text', 'label', 'source']

        if all(col in df.columns for col in required_cols):

            print(
                f"  ✓ Research dataset already exists "
                f"({len(df):,} rows)"
            )

            return df

    print("\n  📥 Loading research abstracts dataset...")

    try:

        ds = load_dataset(
            "NicolaiSivesind/ChatGPT-Research-Abstracts"
        )

        rows = []

        for split in ds.keys():

            for item in tqdm(
                ds[split],
                desc=f"  Research/{split}",
                ncols=70
            ):

                # -----------------------------------------
                # HUMAN ABSTRACT
                # -----------------------------------------

                real_text = clean_text(
                    item.get('real_abstract', '')
                )

                if len(real_text.split()) >= MIN_WORDS:

                    rows.append({
                        'text': real_text[:MAX_TEXT_LENGTH],
                        'label': 0,
                        'source': 'research_human'
                    })

                # -----------------------------------------
                # AI ABSTRACT
                # -----------------------------------------

                fake_text = clean_text(
                    item.get('generated_abstract', '')
                )

                if len(fake_text.split()) >= MIN_WORDS:

                    rows.append({
                        'text': fake_text[:MAX_TEXT_LENGTH],
                        'label': 1,
                        'source': 'research_ai'
                    })

        # -------------------------------------------------
        # DataFrame
        # -------------------------------------------------

        df = pd.DataFrame(rows)

        # Remove duplicates
        df = df.drop_duplicates(
            subset=['text']
        )

        # Balance
        human_df = df[df['label'] == 0]
        ai_df    = df[df['label'] == 1]

        human_df = human_df.sample(
            min(limit_per_class, len(human_df)),
            random_state=42
        )

        ai_df = ai_df.sample(
            min(limit_per_class, len(ai_df)),
            random_state=42
        )

        df = pd.concat([
            human_df,
            ai_df
        ])

        # Shuffle
        df = df.sample(
            frac=1,
            random_state=42
        ).reset_index(drop=True)

        # Save
        df.to_csv(save_path, index=False)

        print(
            f"  ✓ Research dataset saved: "
            f"{len(df):,} rows"
        )

        return df

    except Exception as e:

        print(f"  ⚠ Research dataset failed: {e}")

        return pd.DataFrame(
            columns=['text', 'label', 'source']
        )


# =========================================================
# LOAD DATASETS
# =========================================================

print("\n[3/4] 🤖 ADVANCED AI DATA COLLECTION")
print("-" * 55)

df_gptwiki = load_gpt_wiki_dataset()

df_research = load_research_dataset()

# =========================================================
# SUMMARY
# =========================================================

print("\n" + "="*60)
print("  ✅ DATASET SUMMARY")
print("="*60)

print(f"\nGPT-Wiki rows:   {len(df_gptwiki):,}")

print(f"Research rows:   {len(df_research):,}")

# =========================================================
# COMBINE
# =========================================================

combined_df = pd.concat([
    df_gptwiki,
    df_research
], ignore_index=True)

# Final deduplication
combined_df = combined_df.drop_duplicates(
    subset=['text']
)

# Shuffle
combined_df = combined_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# =========================================================
# SAVE MASTER DATASET
# =========================================================

master_path = os.path.join(
    PROCESSED_DIR,
    'master_ai_detection_dataset.csv'
)

combined_df.to_csv(
    master_path,
    index=False
)

# =========================================================
# FINAL STATS
# =========================================================

print("\n" + "="*60)
print("  🚀 MASTER DATASET READY")
print("="*60)

print(f"\nTotal samples: {len(combined_df):,}")

print("\nLabel distribution:")
print(combined_df['label'].value_counts())

print("\nSource distribution:")
print(combined_df['source'].value_counts())

print(f"\n📁 Saved to:")
print(master_path)

print("\n✅ Ready for advanced ML + Transformer training!")


[3/4] 🤖 ADVANCED AI DATA COLLECTION
-------------------------------------------------------
  ✓ GPT-Wiki already exists (100,000 rows)
  ✓ Research dataset already exists (19,984 rows)

  ✅ DATASET SUMMARY

GPT-Wiki rows:   100,000
Research rows:   19,984

  🚀 MASTER DATASET READY

Total samples: 119,984

Label distribution:
label
0    59998
1    59986
Name: count, dtype: int64

Source distribution:
source
gptwiki_human     50000
gptwiki_ai        50000
research_human     9998
research_ai        9986
Name: count, dtype: int64

📁 Saved to:
e:\ADD\integricheck_project\data\processed\master_ai_detection_dataset.csv

✅ Ready for advanced ML + Transformer training!


## Cell 8: Student Essays — PERSUADE Corpus
**Kya karta hai:** Real student essays download karta hai — AI detection ke liye critical.

### PERSUADE Corpus Kya Hai?
- **Full name:** Persuasive Essays for Rating, Selecting, and Understanding Automated Essay Scoring
- **Size:** ~25,000 real student essays (grades 6-12)
- **Topics:** Argumentative, persuasive, narrative essays
- **Why important:** AI detection model sirf abstract ya Wikipedia pe train ho toh student writing style nahi samjhega

### Fallback:
Agar PERSUADE fail ho toh `WritingPrompts` dataset use hota hai (human creative writing)

### Label:
Yeh saare `label=0` (human written) hain — inse model seekhega ki real student writing kaisi hoti hai

In [14]:
# CELL — ADVANCED HUMAN ESSAY COLLECTION
# Kyu:
#   Real human writing collect karna
#   AI detector ko strong human writing patterns sikhana
#
# Expected output:
#   - writing_prompts_human.csv
#   - balanced human essay corpus

from datasets import load_dataset
import pandas as pd
import numpy as np
import os
import re
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

MAX_TEXT_LENGTH = 4000
MIN_WORDS = 80

TARGET_SAMPLES = 20000

# =========================================================
# CLEANING FUNCTION
# =========================================================

def clean_text(text):

    if not isinstance(text, str):
        return ""

    text = text.replace("\n", " ")
    text = text.replace("\r", " ")

    # Remove URLs
    text = re.sub(r"http\\S+", " ", text)

    # Remove extra spaces
    text = re.sub(r"\\s+", " ", text)

    # Remove broken unicode
    text = text.encode("ascii", "ignore").decode()

    return text.strip()


# =========================================================
# WRITING PROMPTS DATASET
# =========================================================

def load_writing_prompts():

    save_path = os.path.join(
        STUDENT_DIR,
        'writing_prompts_human.csv'
    )

    # -----------------------------------------------------
    # Resume Support
    # -----------------------------------------------------

    if os.path.exists(save_path):

        df = pd.read_csv(save_path)

        required_cols = ['text', 'label', 'source']

        if all(col in df.columns for col in required_cols):

            print(
                f"  ✓ WritingPrompts already exists "
                f"({len(df):,} rows)"
            )

            return df

    print("\n  📥 Loading WritingPrompts dataset...")

    try:

        ds = load_dataset(
            "euclaise/writingprompts"
        )

        rows = []

        for split in ds.keys():

            split_data = ds[split]

            for item in tqdm(
                split_data,
                desc=f"  WritingPrompts/{split}",
                ncols=70
            ):

                # Different versions may use different keys
                text = (
                    item.get('story', '') or
                    item.get('text', '') or
                    item.get('content', '')
                )

                text = clean_text(text)

                # Quality filter
                if len(text.split()) < MIN_WORDS:
                    continue

                rows.append({
                    'text': text[:MAX_TEXT_LENGTH],
                    'label': 0,
                    'source': 'writingprompts_human'
                })

                # Memory protection
                if len(rows) >= TARGET_SAMPLES:
                    break

            if len(rows) >= TARGET_SAMPLES:
                break

        # -------------------------------------------------
        # DataFrame
        # -------------------------------------------------

        df = pd.DataFrame(rows)

        # Remove duplicates
        df = df.drop_duplicates(
            subset=['text']
        )

        # Shuffle
        df = df.sample(
            frac=1,
            random_state=42
        ).reset_index(drop=True)

        # Save
        df.to_csv(save_path, index=False)

        print(
            f"  ✓ WritingPrompts saved: "
            f"{len(df):,} samples"
        )

        return df

    except Exception as e:

        print(f"  ⚠ WritingPrompts failed: {e}")

        return pd.DataFrame(
            columns=['text', 'label', 'source']
        )


# =========================================================
# MAIN EXECUTION
# =========================================================

print("\n[4/4] 🎓 HUMAN ESSAY COLLECTION")
print("-" * 55)

df_student = load_writing_prompts()

# =========================================================
# SUMMARY
# =========================================================

print("\n" + "="*60)
print("  ✅ HUMAN ESSAY DATASET READY")
print("="*60)

print(f"\nTotal essays: {len(df_student):,}")

print("\nSource distribution:")
print(df_student['source'].value_counts())

print(f"\nAverage words per essay:")

avg_words = int(
    df_student['text']
    .apply(lambda x: len(str(x).split()))
    .mean()
)

print(avg_words)

print("\n✅ Ready for master dataset merge!")


[4/4] 🎓 HUMAN ESSAY COLLECTION
-------------------------------------------------------
  ✓ WritingPrompts already exists (19,999 rows)

  ✅ HUMAN ESSAY DATASET READY

Total essays: 19,999

Source distribution:
source
writingprompts_human    19999
Name: count, dtype: int64

Average words per essay:
460

✅ Ready for master dataset merge!


## Cell 9: Master Corpus Build (Plagiarism Database)
**Kya karta hai:** Arxiv + Wikipedia + Student essays ko combine karke ek **Master Corpus** banata hai — yahi hamara plagiarism detection database hai.

### Kaise Kaam Karta Hai:
1. Teeno sources se rows collect karo
2. Duplicate texts remove karo (`drop_duplicates`)
3. Too-short documents filter karo (< 200 chars)
4. CSV mein save karo

### Yeh Corpus Kya Karta Hai?
Jab koi document check karte hain, **har sentence is corpus se compare hoti hai**. Jitna bada corpus, utni zyada chances ki plagiarized content pakad aaye.

### Expected Output:
- Arxiv: ~2,000+ rows
- Wikipedia: ~1,400+ rows
- Student essays: ~5,000 rows (sample)
- **Total: 8,000–30,000 documents**

In [15]:
# CELL — FINAL MASTER CORPUS BUILDER (FIXED + ADVANCED)
# Kyu:
#   Saare plagiarism-search documents ko ek corpus me merge karna
#
# Includes:
#   - arXiv papers
#   - Wikipedia articles
#   - Student essays
#
# Fixes:
#   - Missing doc_id issue
#   - Missing title/text issue
#   - Duplicate handling
#   - Safe CSV generation
#
# Expected output:
#   - master_corpus.csv
#   - Clean plagiarism corpus

import pandas as pd
import numpy as np
import os

print("\n[FINAL] 🔧 BUILDING MASTER CORPUS")
print("-" * 60)

# =========================================================
# STORAGE
# =========================================================

corpus_rows = []

# =========================================================
# ARXIV PAPERS
# =========================================================

print("\n  📚 Adding Arxiv papers...")

arxiv_start = len(corpus_rows)

for idx, p in enumerate(all_arxiv):

    text = str(p.get('text', ''))

    if len(text.split()) < 80:
        continue

    corpus_rows.append({

        'doc_id': p.get(
            'doc_id',
            f"arxiv_{idx:06d}"
        ),

        'title': str(
            p.get('title', 'Arxiv Paper')
        )[:250],

        'text': text[:5000],

        'source': 'arxiv',

        'category': str(
            p.get('category', 'cs')
        ),
    })

print(
    f"    ✓ Arxiv added: "
    f"{len(corpus_rows)-arxiv_start:,}"
)

# =========================================================
# WIKIPEDIA ARTICLES
# =========================================================

print("\n  📖 Adding Wikipedia articles...")

wiki_start = len(corpus_rows)

for idx, a in enumerate(all_wiki):

    text = str(a.get('text', ''))

    if len(text.split()) < 100:
        continue

    corpus_rows.append({

        # FIXED SAFE DOC_ID
        'doc_id': a.get(
            'doc_id',
            f"wiki_{idx:06d}"
        ),

        'title': str(
            a.get('title', 'Wikipedia Article')
        )[:250],

        'text': text[:5000],

        'source': 'wikipedia',

        'category': str(
            a.get('category', 'encyclopedia')
        ),
    })

print(
    f"    ✓ Wikipedia added: "
    f"{len(corpus_rows)-wiki_start:,}"
)

# =========================================================
# STUDENT ESSAYS
# =========================================================

print("\n  🎓 Adding student essays...")

stu_start = len(corpus_rows)

if not df_student.empty:

    # Sample for memory efficiency
    df_stu_sample = df_student.sample(
        min(len(df_student), 5000),
        random_state=42
    )

    for idx, row in df_stu_sample.iterrows():

        text = str(row.get('text', ''))

        if len(text.split()) < 80:
            continue

        corpus_rows.append({

            'doc_id': f"student_{idx:06d}",

            'title': str(
                row.get(
                    'prompt',
                    'Student Essay'
                )
            )[:250],

            'text': text[:5000],

            'source': 'student_essay',

            'category': 'academic',
        })

print(
    f"    ✓ Student essays added: "
    f"{len(corpus_rows)-stu_start:,}"
)

# =========================================================
# CREATE DATAFRAME
# =========================================================

print("\n  🧠 Creating DataFrame...")

master_df = pd.DataFrame(corpus_rows)

print(f"    Initial rows: {len(master_df):,}")

# =========================================================
# CLEANING
# =========================================================

print("\n  🧹 Cleaning corpus...")

# Remove NaN
master_df = master_df.dropna(
    subset=['text']
)

# Convert to string
master_df['text'] = master_df['text'].astype(str)

# Remove duplicates
before_dup = len(master_df)

master_df = master_df.drop_duplicates(
    subset='text',
    keep='first'
)

after_dup = len(master_df)

print(
    f"    Removed duplicates: "
    f"{before_dup-after_dup:,}"
)

# Remove short texts
before_short = len(master_df)

master_df = master_df[
    master_df['text']
    .str.split()
    .str.len() > 80
]

after_short = len(master_df)

print(
    f"    Removed short docs: "
    f"{before_short-after_short:,}"
)

# Reset index
master_df = master_df.reset_index(drop=True)

# =========================================================
# SAVE
# =========================================================

master_path = os.path.join(
    PROCESSED_DIR,
    'master_corpus.csv'
)

master_df.to_csv(
    master_path,
    index=False
)

# =========================================================
# FILE SIZE
# =========================================================

file_size_mb = (
    os.path.getsize(master_path)
    / 1024 / 1024
)

# =========================================================
# SUMMARY
# =========================================================

print("\n" + "="*65)
print("  ✅ MASTER CORPUS COMPLETE")
print("="*65)

print(f"\n📊 Total Documents: {len(master_df):,}")

print("\n📚 Source Breakdown:")

for src, count in master_df['source'].value_counts().items():

    pct = count / len(master_df) * 100

    print(
        f"  {src:20s}: "
        f"{count:>7,} "
        f"({pct:.1f}%)"
    )

print("\n📂 Category Breakdown:")

for cat, count in master_df['category'].value_counts().head(10).items():

    print(
        f"  {cat:20s}: "
        f"{count:>7,}"
    )

# =========================================================
# TEXT LENGTH STATS
# =========================================================

word_counts = master_df['text'].apply(
    lambda x: len(str(x).split())
)

print("\n📝 Text Statistics:")

print(f"  Average words: {int(word_counts.mean())}")

print(f"  Max words:     {word_counts.max()}")

print(f"  Min words:     {word_counts.min()}")

# =========================================================
# SAVE INFO
# =========================================================

print(f"\n📁 Saved to:")
print(master_path)

print(f"\n💾 File size:")
print(f"{file_size_mb:.1f} MB")

print("\n🚀 MASTER CORPUS READY FOR:")
print("  ✓ Plagiarism Detection")
print("  ✓ Semantic Search")
print("  ✓ Similarity Matching")
print("  ✓ Embedding Generation")
print("  ✓ RAG Pipeline")

print("\n" + "="*65)


[FINAL] 🔧 BUILDING MASTER CORPUS
------------------------------------------------------------

  📚 Adding Arxiv papers...
    ✓ Arxiv added: 2,364

  📖 Adding Wikipedia articles...
    ✓ Wikipedia added: 1,220

  🎓 Adding student essays...
    ✓ Student essays added: 5,000

  🧠 Creating DataFrame...
    Initial rows: 8,584

  🧹 Cleaning corpus...
    Removed duplicates: 515
    Removed short docs: 0

  ✅ MASTER CORPUS COMPLETE

📊 Total Documents: 8,069

📚 Source Breakdown:
  student_essay       :   5,000 (62.0%)
  arxiv               :   1,944 (24.1%)
  wikipedia           :   1,125 (13.9%)

📂 Category Breakdown:
  academic            :   5,000
  general             :   1,095
  cs.AI               :     397
  cs.IR               :     372
  stat.ML             :     358
  cs.CL               :     319
  cs.CV               :     318
  cs.LG               :     180
  encyclopedia        :      30

📝 Text Statistics:
  Average words: 409
  Max words:     896
  Min words:     100

📁 Save

## Cell 10: AI Detection Dataset Build
**Kya karta hai:** HC3 + GPT-wiki + RAID + Student essays ko combine karke **AI Detection training data** banata hai.

### Class Balance:
- **label=0** → Human written text
- **label=1** → AI generated text

Achha AI detection model ke liye roughly equal human aur AI samples chahiye.

### Max rows per source: 15,000
Taaki ek source dominant na ho — diverse training data = robust model.

In [17]:
import pandas as pd
import numpy as np
import os

print("\n🤖 BUILDING ADVANCED AI DETECTION DATASET")
print("-" * 60)

ai_rows = []

# =========================================================
# SAFE ADD FUNCTION
# =========================================================

def add_to_ai_dataset(df, max_rows=15000, label_col='label', name="dataset"):
    
    if df is None or df.empty:
        print(f"⚠ Skipping {name} (empty or not found)")
        return

    sample_size = min(len(df), max_rows)

    df_s = df.sample(sample_size, random_state=42)

    added = 0

    for _, row in df_s.iterrows():

        text = str(row.get('text', '')).strip()

        # Quality filter
        if len(text.split()) < 40:
            continue

        ai_rows.append({
            'text': text[:3000],
            'label': int(row.get(label_col, 0)),
            'source': str(row.get('source', 'unknown')),
        })

        added += 1

    print(f"✔ Added {added:,} samples from {name}")


# =========================================================
# SAFE DATASET LOADING CHECKS
# =========================================================

print("\n📥 Adding datasets...")

# GPT-Wiki
if "df_gptwiki" in globals():
    add_to_ai_dataset(df_gptwiki, max_rows=30000, name="GPT-Wiki")
else:
    print("⚠ df_gptwiki not found")

# RAiD dataset (FIXED ERROR)
if "df_raid" in globals():
    add_to_ai_dataset(df_raid, max_rows=20000, name="RAiD")
else:
    print("⚠ df_raid not found — skipping")

# Student essays
if "df_student" in globals() and not df_student.empty:
    df_stu_ai = df_student.copy()
    df_stu_ai['label'] = 0
    add_to_ai_dataset(df_stu_ai, max_rows=10000, name="Student Essays")
else:
    print("⚠ df_student not found — skipping")


# =========================================================
# CREATE DATAFRAME
# =========================================================

print("\n🧠 Creating DataFrame...")

ai_df = pd.DataFrame(ai_rows)

print(f"Initial rows: {len(ai_df):,}")

if ai_df.empty:
    raise ValueError("❌ No data collected. Check input datasets.")

# =========================================================
# CLEANING
# =========================================================

before_dup = len(ai_df)

ai_df = ai_df.dropna(subset=['text'])
ai_df = ai_df.drop_duplicates(subset='text', keep='first')

after_dup = len(ai_df)

print(f"🧹 Removed duplicates: {before_dup - after_dup:,}")

# =========================================================
# BALANCING
# =========================================================

print("\n⚖ Balancing dataset...")

human_df = ai_df[ai_df['label'] == 0]
ai_only_df = ai_df[ai_df['label'] == 1]

print(f"Human samples: {len(human_df):,}")
print(f"AI samples:    {len(ai_only_df):,}")

if len(human_df) == 0 or len(ai_only_df) == 0:
    raise ValueError("❌ One class is empty. Cannot balance dataset.")

target = min(len(human_df), len(ai_only_df))

human_balanced = human_df.sample(target, random_state=42)
ai_balanced = ai_only_df.sample(target, random_state=42)

ai_df = pd.concat([human_balanced, ai_balanced])

ai_df = ai_df.sample(frac=1, random_state=42).reset_index(drop=True)

# =========================================================
# SAVE
# =========================================================

if "PROCESSED_DIR" not in globals():
    PROCESSED_DIR = "."

ai_path = os.path.join(PROCESSED_DIR, 'ai_detection_data.csv')

ai_df.to_csv(ai_path, index=False)

# =========================================================
# FINAL STATS
# =========================================================

human_n = len(ai_df[ai_df.label == 0])
ai_n = len(ai_df[ai_df.label == 1])

file_size_mb = os.path.getsize(ai_path) / 1024 / 1024

avg_words = int(ai_df['text'].apply(lambda x: len(str(x).split())).mean())

# =========================================================
# SUMMARY
# =========================================================

print("\n" + "="*65)
print("  ✅ AI DETECTION DATASET COMPLETE")
print("="*65)

print(f"\n📊 Total Samples: {len(ai_df):,}")

print(f"\nHuman (0): {human_n:,} ({human_n/len(ai_df)*100:.1f}%)")
print(f"AI (1):    {ai_n:,} ({ai_n/len(ai_df)*100:.1f}%)")

print("\n📝 Average words per sample:", avg_words)

print(f"\n💾 File size: {file_size_mb:.1f} MB")

print(f"\n📁 Saved at: {ai_path}")

print("\n🚀 READY FOR:")
print("  ✓ TF-IDF")
print("  ✓ Logistic Regression")
print("  ✓ XGBoost")
print("  ✓ RoBERTa Fine-tuning")
print("  ✓ Transformer Training")

print("\n" + "="*65)


🤖 BUILDING ADVANCED AI DETECTION DATASET
------------------------------------------------------------

📥 Adding datasets...
✔ Added 30,000 samples from GPT-Wiki
⚠ df_raid not found — skipping
✔ Added 10,000 samples from Student Essays

🧠 Creating DataFrame...
Initial rows: 40,000
🧹 Removed duplicates: 0

⚖ Balancing dataset...
Human samples: 24,906
AI samples:    15,094

  ✅ AI DETECTION DATASET COMPLETE

📊 Total Samples: 30,188

Human (0): 15,094 (50.0%)
AI (1):    15,094 (50.0%)

📝 Average words per sample: 206

💾 File size: 35.2 MB

📁 Saved at: e:\ADD\integricheck_project\data\processed\ai_detection_data.csv

🚀 READY FOR:
  ✓ TF-IDF
  ✓ Logistic Regression
  ✓ XGBoost
  ✓ RoBERTa Fine-tuning
  ✓ Transformer Training



## ✅ Step 2 Complete!

### Files Generated:
| File | Purpose |
|------|---------|
| `data/raw/arxiv/all_papers.json` | 2,400+ arxiv papers |
| `data/raw/wikipedia/wikipedia_articles.json` | 1,500+ wiki articles |
| `data/raw/ai_datasets/hc3_dataset.csv` | HC3 human vs ChatGPT |
| `data/raw/ai_datasets/gpt_wiki_dataset.csv` | GPT wiki intros |
| `data/raw/ai_datasets/raid_dataset.csv` | Multi-model AI data |
| `data/raw/student_essays/persuade_essays.csv` | Real student essays |
| `data/processed/master_corpus.csv` | **Plagiarism database** |
| `data/processed/ai_detection_data.csv` | **AI detection training data** |

### ▶️ Next Step:
`step3_plagiarism_engine_v2.ipynb` chalaao — plagiarism models train hoge!

In [18]:
print("\n" + "="*60)
print("  STEP 2 COMPLETE ✅")
print("="*60)

# Quick sanity check
master_df_check = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_corpus.csv'))
ai_df_check     = pd.read_csv(os.path.join(PROCESSED_DIR, 'ai_detection_data.csv'))

print(f"\n  Plagiarism corpus:    {len(master_df_check):,} documents")
print(f"  AI detection data:    {len(ai_df_check):,} samples")
print(f"\n  ▶ Next: Run step3_plagiarism_engine_v2.ipynb")
print("="*60)



  STEP 2 COMPLETE ✅

  Plagiarism corpus:    8,069 documents
  AI detection data:    30,188 samples

  ▶ Next: Run step3_plagiarism_engine_v2.ipynb
